In [16]:
import pandas as pd

DRAINED_FILE   = 'Sketch-Clay/Drained clay data.xlsx'
UNDRAINED_FILE = 'Sketch-Clay/Undrained clay data.xlsx'

SHEETS = [
    'Normally consolidated (NC)',
    'Critical state (CS)',
    'Overconsolidated (OC)',
]

# --- Read raw Excel sheets ---
drained_NC = pd.read_excel(DRAINED_FILE, sheet_name=SHEETS[0])
drained_CS = pd.read_excel(DRAINED_FILE, sheet_name=SHEETS[1])
drained_OC = pd.read_excel(DRAINED_FILE, sheet_name=SHEETS[2])

undrained_NC = pd.read_excel(UNDRAINED_FILE, sheet_name=SHEETS[0])
undrained_CS = pd.read_excel(UNDRAINED_FILE, sheet_name=SHEETS[1])
undrained_OC = pd.read_excel(UNDRAINED_FILE, sheet_name=SHEETS[2])

# --- Cleaning helper ---

def clean_df(df, name=None):
    """Attempt to fix multi-row headers and remove non-numeric top rows.

    Strategy:
    - Drop completely empty columns.
    - Look at the top 3 rows and mark any row as a header-row if >=60% of its
      non-empty cells are non-numeric.
    - If header rows are found, construct new column names by joining the
      non-empty values from those rows per column (joined with ' - ').
    - Remove the header rows from the DataFrame, coerce remaining cells to
      numeric where possible, and drop rows that are entirely empty (all NaN).
    """
    df = df.copy()
    # drop empty columns
    df.dropna(axis=1, how='all', inplace=True)

    max_check = min(3, len(df))
    header_rows = []
    for i in range(max_check):
        row = df.iloc[i].fillna("").astype(str)
        non_empty = [s for s in row if s.strip() != ""]
        if len(non_empty) == 0:
            continue
        non_numeric = 0
        total = 0
        for s in non_empty:
            total += 1
            try:
                float(s)
            except Exception:
                non_numeric += 1
        frac = non_numeric / max(1, total)
        if frac >= 0.6:
            header_rows.append(i)

    if header_rows:
        last_header = max(header_rows)
        new_cols = []
        for col in df.columns:
            parts = []
            for r in header_rows:
                val = df.iloc[r][col]
                if pd.isna(val):
                    continue
                s = str(val).strip()
                if s != "":
                    parts.append(s)
            new_cols.append(' - '.join(parts) if parts else str(col))
        df.columns = new_cols
        df = df.iloc[last_header + 1 :].reset_index(drop=True)
    else:
        # fallback: sanitize existing column names
        df.columns = [str(c).strip() for c in df.columns]

    # Try to coerce all columns to numeric where possible
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # Drop rows that are completely empty (all NaN)
    df.dropna(how='all', inplace=True)
    df.reset_index(drop=True, inplace=True)
    return df

# Apply cleaning to all dataframes
for var in ['drained_NC', 'drained_CS', 'drained_OC', 'undrained_NC', 'undrained_CS', 'undrained_OC']:
    globals()[var] = clean_df(globals()[var], name=var)

print('Drained NC:',   drained_NC.shape)
print('Drained CS:',   drained_CS.shape)
print('Drained OC:',   drained_OC.shape)
print('Undrained NC:', undrained_NC.shape)
print('Undrained CS:', undrained_CS.shape)
print('Undrained OC:', undrained_OC.shape)


Drained NC: (501, 5)
Drained CS: (501, 5)
Drained OC: (501, 5)
Undrained NC: (1999, 5)
Undrained CS: (1999, 5)
Undrained OC: (1999, 5)


C:\Users\pt620\AppData\Local\Temp\ipykernel_39720\64585831.py:42: FutureWarning:

Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`

C:\Users\pt620\AppData\Local\Temp\ipykernel_39720\64585831.py:42: FutureWarning:

Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`

C:\Users\pt620\AppData\Local\Temp\ipykernel_39720\64585831.py:42: FutureWarning:

Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`

C:\U

In [17]:
# Preview drained data
print('=== Drained NC ===')
display(drained_NC.head())

print('=== Drained CS ===')
display(drained_CS.head())

print('=== Drained OC ===')
display(drained_OC.head())

=== Drained NC ===


,ε_ax - (%),q - (kPa),p' - (kPa),u - (kPa),e - (-)
0,0.0,0.000000,1095.000000,0.0000,0.600000
1,0.1,40.322143,1108.440714,0.1564,0.597498
2,0.2,76.314159,1120.438053,0.3067,0.595093
3,0.3,110.851252,1131.950417,0.4559,0.592706
4,0.4,143.569691,1142.856564,0.6037,0.590341


=== Drained CS ===


,ε_ax - (%),q - (kPa),p' - (kPa),u - (kPa),e - (-)
0,0.0,0.000000,650.000000,0.0000,0.600000
1,0.1,28.994531,659.664844,0.0659,0.598946
2,0.2,53.780178,667.926726,0.1374,0.597802
3,0.3,78.600466,676.200155,0.2071,0.596686
4,0.4,103.420754,684.473585,0.2745,0.595608


=== Drained OC ===


,ε_ax - (%),q - (kPa),p' - (kPa),u - (kPa),e - (-)
0,0.0,0.000000,300.000000,0.0000,0.600000
1,0.1,30.258928,310.086309,0.0816,0.598694
2,0.2,56.967151,318.989050,0.1603,0.597435
3,0.3,83.883221,327.961074,0.2408,0.596147
4,0.4,111.128380,337.042793,0.3225,0.594840


In [18]:
# Preview undrained data
print('=== Undrained NC ===')
display(undrained_NC.head())

print('=== Undrained CS ===')
display(undrained_CS.head())

print('=== Undrained OC ===')
display(undrained_OC.head())

=== Undrained NC ===


,ε_ax - (%),q - (kPa),p' - (kPa),u - (kPa),e - (-)
0,0.00,0.000000,1090.00,0.000000,0.6
1,0.03,6.716893,1090.00,6.938964,0.6
2,0.04,20.949155,1089.70,12.283052,0.6
3,0.05,34.980498,1089.09,17.570166,0.6
4,0.06,48.684484,1088.48,22.748161,0.6


=== Undrained CS ===


,ε_ax - (%),q - (kPa),p' - (kPa),u - (kPa),e - (-)
0,0.00,0.000000,650.00,0.000000,0.6
1,0.03,10.281454,651.96,1.467151,0.6
2,0.04,15.527835,651.96,3.215945,0.6
3,0.05,20.770753,651.96,4.963584,0.6
4,0.06,26.006743,651.95,6.718914,0.6


=== Undrained OC ===


,ε_ax - (%),q - (kPa),p' - (kPa),u - (kPa),e - (-)
0,0.00,0.000000,300.00,0.000000,0.6
1,0.03,2.404087,300.35,0.451362,0.6
2,0.04,5.045464,300.35,1.331821,0.6
3,0.05,7.686841,300.35,2.212280,0.6
4,0.06,10.329951,300.36,3.083317,0.6


In [20]:
# Interactive 3x2 grid of plots A-F (each test plotted separately)
import plotly.graph_objs as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output

# Helper to robustly find column names
def find_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    # try substring match
    for col in df.columns:
        lc = str(col).lower()
        for c in candidates:
            if c.lower() in lc:
                return col
    return None

# Prepare datasets: compute du and volumetric strain ε_vol per source DF
def prepare_source(df):
    df = df.copy()
    col_eps = find_col(df, ['ε_ax - (%)', 'eps_ax', 'ε_ax', 'axial'])
    col_q   = find_col(df, ['q - (kPa)', 'q'])
    col_p   = find_col(df, ["p' - (kPa)", "p'", 'p - (kPa)', 'p'])
    col_u   = find_col(df, ['u - (kPa)', 'u'])
    col_e   = find_col(df, ['e - (-)', 'e'])

    if col_eps is None or col_q is None or col_p is None or col_u is None or col_e is None:
        raise KeyError('Expected columns not found; available: ' + ','.join(df.columns.astype(str)))

    # rename for convenience
    df = df.rename(columns={col_eps: 'eps_ax', col_q: 'q', col_p: 'p', col_u: 'u', col_e: 'e'})

    # compute du (relative to first row in this source) and eps_vol
    if len(df) > 0:
        df['du'] = df['u'] - df['u'].iloc[0]
        e0 = df['e'].iloc[0]
        df['eps_vol'] = - (df['e'] - e0) / (1 + e0)
    else:
        df['du'] = pd.Series(dtype=float)
        df['eps_vol'] = pd.Series(dtype=float)
    return df

# Prepare individual datasets (do not concatenate) so each test is plotted separately
srcs_drained = [
    ('NC', prepare_source(drained_NC)),
    ('CS', prepare_source(drained_CS)),
    ('OC', prepare_source(drained_OC)),
]
srcs_undrained = [
    ('NC', prepare_source(undrained_NC)),
    ('CS', prepare_source(undrained_CS)),
    ('OC', prepare_source(undrained_OC)),
]

# line style mapping for NC/CS/OC
style_map = {
    'NC': {'dash': 'solid'},
    'CS': {'dash': 'dash'},
    'OC': {'dash': 'dot'},
}

# plotting function with 3 rows x 2 cols layout
def make_figure(mode='both'):
    show_drained = mode in ('both', 'drained')
    show_undrained = mode in ('both', 'undrained')

    specs = [
        [{}, {}],
        [{}, {"secondary_y": True}],
        [{"type": "scene"}, {}],
    ]

    fig = make_subplots(rows=3, cols=2, specs=specs,
                        subplot_titles=(
                            "A: q - p'",     # 1,1
                            "B: q - ε_ax",   # 1,2
                            "C: e - p'",     # 2,1
                            "D: du & ε_vol - ε_ax", # 2,2 (secondary_y)
                            "E: 3D q - p' - e", # 3,1 (scene)
                            "F: q/p' - ε_ax"    # 3,2
                        ))

    drained_color = 'blue'
    undrained_color = 'red'

    # A: q vs p' (row1,col1) - plot each test separately
    if show_drained:
        for tag, df in srcs_drained:
            fig.add_trace(go.Scatter(x=df['p'], y=df['q'], mode='lines', name=f'drained {tag}', line=dict(color=drained_color, dash=style_map[tag]['dash'])), row=1, col=1)
    if show_undrained:
        for tag, df in srcs_undrained:
            fig.add_trace(go.Scatter(x=df['p'], y=df['q'], mode='lines', name=f'undrained {tag}', line=dict(color=undrained_color, dash=style_map[tag]['dash'])), row=1, col=1)

    # B: q vs eps_ax (row1,col2)
    if show_drained:
        for tag, df in srcs_drained:
            fig.add_trace(go.Scatter(x=df['eps_ax'], y=df['q'], mode='lines', name=f'drained {tag}', line=dict(color=drained_color, dash=style_map[tag]['dash'])), row=1, col=2)
    if show_undrained:
        for tag, df in srcs_undrained:
            fig.add_trace(go.Scatter(x=df['eps_ax'], y=df['q'], mode='lines', name=f'undrained {tag}', line=dict(color=undrained_color, dash=style_map[tag]['dash'])), row=1, col=2)

    # C: e vs p' (row2,col1)
    if show_drained:
        for tag, df in srcs_drained:
            fig.add_trace(go.Scatter(x=df['p'], y=df['e'], mode='lines', name=f'drained {tag}', line=dict(color=drained_color, dash=style_map[tag]['dash'])), row=2, col=1)
    if show_undrained:
        for tag, df in srcs_undrained:
            fig.add_trace(go.Scatter(x=df['p'], y=df['e'], mode='lines', name=f'undrained {tag}', line=dict(color=undrained_color, dash=style_map[tag]['dash'])), row=2, col=1)

    # D: du and eps_vol vs eps_ax on row2,col2. If only drained -> eps_vol traces; only undrained -> du traces; both -> left eps_vol (drained), right du (undrained)
    if mode == 'drained':
        for tag, df in srcs_drained:
            fig.add_trace(go.Scatter(x=df['eps_ax'], y=df['eps_vol'], mode='lines', name=f'ε_vol drained {tag}', line=dict(color=drained_color, dash=style_map[tag]['dash'])), row=2, col=2, secondary_y=False)
        fig.update_yaxes(title_text='ε_vol', row=2, col=2, secondary_y=False)
    elif mode == 'undrained':
        for tag, df in srcs_undrained:
            fig.add_trace(go.Scatter(x=df['eps_ax'], y=df['du'], mode='lines', name=f'du undrained {tag}', line=dict(color=undrained_color, dash=style_map[tag]['dash'])), row=2, col=2, secondary_y=False)
        fig.update_yaxes(title_text='du (kPa)', row=2, col=2, secondary_y=False)
    else:  # both
        for tag, df in srcs_drained:
            fig.add_trace(go.Scatter(x=df['eps_ax'], y=df['eps_vol'], mode='lines', name=f'ε_vol drained {tag}', line=dict(color=drained_color, dash=style_map[tag]['dash'])), row=2, col=2, secondary_y=False)
        for tag, df in srcs_undrained:
            fig.add_trace(go.Scatter(x=df['eps_ax'], y=df['du'], mode='lines', name=f'du undrained {tag}', line=dict(color=undrained_color, dash=style_map[tag]['dash'])), row=2, col=2, secondary_y=True)
        fig.update_yaxes(title_text='ε_vol', row=2, col=2, secondary_y=False)
        fig.update_yaxes(title_text='du (kPa)', row=2, col=2, secondary_y=True)

    # E: 3D plot q - p' - e on row3,col1 (scene)
    if show_drained:
        for tag, df in srcs_drained:
            fig.add_trace(go.Scatter3d(x=df['q'], y=df['p'], z=df['e'], mode='lines', name=f'drained {tag}', line=dict(color=drained_color, dash=style_map[tag]['dash'])), row=3, col=1)
    if show_undrained:
        for tag, df in srcs_undrained:
            fig.add_trace(go.Scatter3d(x=df['q'], y=df['p'], z=df['e'], mode='lines', name=f'undrained {tag}', line=dict(color=undrained_color, dash=style_map[tag]['dash'])), row=3, col=1)

    # F: q/p' vs eps_ax on row3,col2; protect against division by zero
    if show_drained:
        for tag, df in srcs_drained:
            denom = df['p'].replace({0: pd.NA})
            y = df['q'] / denom
            fig.add_trace(go.Scatter(x=df['eps_ax'], y=y, mode='lines', name=f'drained {tag}', line=dict(color=drained_color, dash=style_map[tag]['dash'])), row=3, col=2)
    if show_undrained:
        for tag, df in srcs_undrained:
            denom = df['p'].replace({0: pd.NA})
            y = df['q'] / denom
            fig.add_trace(go.Scatter(x=df['eps_ax'], y=y, mode='lines', name=f'undrained {tag}', line=dict(color=undrained_color, dash=style_map[tag]['dash'])), row=3, col=2)

    # Axis titles
    fig.update_xaxes(title_text="p' (kPa)", row=1, col=1)
    fig.update_yaxes(title_text='q (kPa)', row=1, col=1)

    fig.update_xaxes(title_text='ε_ax (%)', row=1, col=2)
    fig.update_yaxes(title_text='q (kPa)', row=1, col=2)

    fig.update_xaxes(title_text="p' (kPa)", row=2, col=1)
    fig.update_yaxes(title_text='e (-)', row=2, col=1)

    fig.update_xaxes(title_text='ε_ax (%)', row=2, col=2)

    fig.update_xaxes(title_text='ε_ax (%)', row=3, col=2)
    fig.update_yaxes(title_text="q/p'", row=3, col=2)

    fig.update_layout(height=1100, width=1200, showlegend=True, title_text='Interactive grid: A-F (3x2)')
    return fig

# interactive selector
mode_selector = widgets.Dropdown(options=[('Both', 'both'), ('Drained only', 'drained'), ('Undrained only', 'undrained')], value='both', description='Mode:')
output = widgets.Output()

def on_mode_change(change):
    if change['name'] == 'value':
        with output:
            clear_output(wait=True)
            fig = make_figure(mode=change['new'])
            fig.show()

mode_selector.observe(on_mode_change)

# initial display
display(mode_selector)
with output:
    fig = make_figure(mode_selector.value)
    fig.show()

display(output)


Dropdown(description='Mode:', options=(('Both', 'both'), ('Drained only', 'drained'), ('Undrained only', 'undr…

Output()